# Checkpointing & Recovery

Make a long-running QAOA optimization survive a mid-run failure by periodically saving its state and resuming from the last checkpoint instead of restarting from zero.

**Objectives:**
- Run a QAOA-style optimization loop locally on the Braket `LocalSimulator` and watch it converge
- Periodically checkpoint optimizer state (parameters + iteration) to a temp-file checkpoint
- Simulate a failure partway through, then restart and demonstrably resume from the last checkpoint
- Map the local pattern to the real `save_job_checkpoint()` / `load_job_checkpoint()` API inside a gated Hybrid Job entry-point
- Reason about the checkpoint-interval vs. overhead trade-off

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

import json
import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

from braket.circuits import Circuit, FreeParameter
from braket.devices import LocalSimulator

# These import fine with no AWS credentials. We only CALL them inside the gated
# entry-point (a real Hybrid Job), never at top level in this notebook.
from braket.aws import AwsQuantumJob
from braket.jobs import save_job_checkpoint, load_job_checkpoint, save_job_result
from braket.jobs.metrics import log_metric

# Use the local simulator by default (free, instant, no credentials).
device = LocalSimulator()

# Determinism for the whole notebook.
np.random.seed(7)
rng = np.random.default_rng(7)

print("Imports OK. Running on LocalSimulator (no AWS, no cost).")

## 1. A small QAOA-style optimization to protect

A job that runs for hours is a job that can fail for hours' worth of reasons: a reclaimed spot instance, a transient device error, a timeout. The thing worth protecting is the *optimizer state* — the parameters it has painstakingly tuned over many iterations.

To have something concrete and fast to checkpoint, we use a minimal **QAOA MaxCut** on a single-edge graph $(0, 1)$. The cost layer is a $ZZ$ rotation by $2\gamma$ on the edge; the mixer is an $R_x(2\beta)$ on each qubit. The objective is the expected cut value — the probability that the two qubits end up in different states, i.e. $P(|01\rangle) + P(|10\rangle)$. We declare $\gamma$ and $\beta$ as `FreeParameter`s so the same compiled circuit is reused every iteration (parametric compilation, the inner loop of every hybrid job).

We run with `shots=0`, which returns *exact* probabilities from the state vector — deterministic, so an uninterrupted run and a resumed run will reach the identical final state and we can assert on it.

In [ ]:
gamma = FreeParameter("gamma")
beta = FreeParameter("beta")


def build_qaoa_circuit():
    """One-layer QAOA for MaxCut on the single edge (0, 1)."""
    c = Circuit().h(0).h(1)          # uniform superposition
    c.zz(0, 1, 2 * gamma)            # cost layer on edge (0, 1)
    c.rx(0, 2 * beta).rx(1, 2 * beta)  # mixer layer
    c.probability()                   # ask for the full probability vector
    return c


qaoa_circuit = build_qaoa_circuit()


def cut_value(params):
    """Expected MaxCut value: P(|01>) + P(|10>). Exact via shots=0."""
    result = device.run(
        qaoa_circuit,
        shots=0,
        inputs={"gamma": float(params[0]), "beta": float(params[1])},
    ).result()
    p = result.values[0]  # probabilities for basis states 00, 01, 10, 11
    return p[1] + p[2]


def neg_cut(params):
    """We minimize the negative cut (equivalently, maximize the cut)."""
    return -cut_value(params)


print("Circuit (reused every iteration via FreeParameter):")
print(qaoa_circuit)
print(f"\nInitial cut value at gamma=beta=0.1: {cut_value(np.array([0.1, 0.1])):.4f}")

## 2. The optimization loop (the uninterrupted reference)

A finite-difference gradient ascent drives $(\gamma, \beta)$ toward the maximum cut. This is the loop a real QAOA job runs hundreds of times. First we run it straight through, with **no failure and no checkpointing**, to establish the reference trajectory and final state that a recovered run must match.

In [ ]:
def fd_gradient(params, eps=1e-3):
    """Central finite-difference gradient of neg_cut."""
    grad = np.zeros(2)
    for i in range(2):
        step = np.zeros(2)
        step[i] = eps
        grad[i] = (neg_cut(params + step) - neg_cut(params - step)) / (2 * eps)
    return grad


def gradient_step(params, lr=0.3):
    return params - lr * fd_gradient(params)


N_ITERS = 20
INITIAL_PARAMS = np.array([0.1, 0.1])

# Uninterrupted reference run.
ref_params = INITIAL_PARAMS.copy()
ref_curve = []
for it in range(N_ITERS):
    ref_params = gradient_step(ref_params)
    ref_curve.append(cut_value(ref_params))

ref_final_cut = cut_value(ref_params)
print(f"Reference run: {N_ITERS} iterations, no failure.")
print(f"Final params (gamma, beta): {np.round(ref_params, 4)}")
print(f"Final cut value: {ref_final_cut:.4f}  (optimum for one edge is 1.0)")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, N_ITERS + 1), ref_curve, "o-", color="#2563eb")
plt.axhline(1.0, ls="--", color="#9ca3af", label="optimum (cut = 1.0)")
plt.xlabel("iteration")
plt.ylabel("cut value")
plt.title("Uninterrupted QAOA convergence (LocalSimulator)")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Checkpointing to a temp file

Now we make the loop *recoverable*. The checkpoint is a tiny JSON blob holding everything needed to resume: the last completed iteration, the current parameters, and the best cut seen so far. We write it to a `tempfile` directory — never a committed file in the repo — so the demo is self-contained and leaves nothing behind.

The key design choice is the **checkpoint interval** `every`: we save every `every` iterations rather than every step, trading recovery granularity against I/O overhead (more on that in section 6).

In [ ]:
CKPT_PATH = os.path.join(tempfile.mkdtemp(prefix="qaoa_ckpt_"), "checkpoint.json")
print(f"Checkpoint file (temp, not committed): {CKPT_PATH}")


def save_checkpoint(iteration, params, best_cut):
    with open(CKPT_PATH, "w") as f:
        json.dump(
            {"iteration": int(iteration), "params": params.tolist(), "best_cut": float(best_cut)},
            f,
        )


def load_checkpoint():
    with open(CKPT_PATH, "r") as f:
        d = json.load(f)
    return d["iteration"], np.array(d["params"]), d["best_cut"]


def checkpoint_exists():
    return os.path.exists(CKPT_PATH)

## 4. Run, fail, and resume

`run_recoverable` is written exactly as a fault-tolerant job loop would be: on entry it looks for a checkpoint and, if one exists, resumes from `last_iteration + 1` with the saved parameters. Otherwise it starts fresh. We deliberately raise a `SimulatedFailure` partway through to stand in for a reclaimed spot instance — note it saves a checkpoint right before failing.

We call it twice: the first call crashes at iteration 12; the second call (same function, no arguments changed) finds the checkpoint and continues from there.

In [ ]:
CHECKPOINT_EVERY = 5


class SimulatedFailure(Exception):
    """Stands in for a reclaimed spot instance / transient device error."""


def run_recoverable(n_iters, fail_at=None):
    # --- Resume logic: this is the whole point of checkpointing ---
    if checkpoint_exists():
        last_it, params, best_cut = load_checkpoint()
        start = last_it + 1
        print(f"  Resuming from checkpoint at iteration {last_it} "
              f"(params={np.round(params, 4)}). Continuing at {start}.")
    else:
        start, params, best_cut = 0, INITIAL_PARAMS.copy(), -1.0
        print("  No checkpoint found. Starting fresh at iteration 0.")

    for it in range(start, n_iters):
        params = gradient_step(params)
        best_cut = max(best_cut, cut_value(params))
        if it % CHECKPOINT_EVERY == 0:
            save_checkpoint(it, params, best_cut)
        if fail_at is not None and it == fail_at:
            save_checkpoint(it, params, best_cut)  # persist progress before dying
            raise SimulatedFailure(f"instance reclaimed at iteration {it}")

    save_checkpoint(n_iters - 1, params, best_cut)  # final checkpoint
    return params, best_cut


print("First attempt (will fail mid-run):")
try:
    run_recoverable(N_ITERS, fail_at=12)
except SimulatedFailure as e:
    print(f"  Caught simulated failure: {e}")

resumed_it, _, _ = load_checkpoint()
print(f"\nCheckpoint on disk after failure: last saved iteration = {resumed_it}")

print("\nSecond attempt (restart -- should resume, not restart):")
resumed_params, resumed_best = run_recoverable(N_ITERS)
resumed_final_cut = cut_value(resumed_params)
final_it, _, _ = load_checkpoint()

### Verify correct resumption

Two things must hold for this to count as a real recovery, not an accidental restart-from-zero:

1. The run **continued past the failure point** — the final checkpoint iteration is `N_ITERS - 1`, well beyond where it crashed (12).
2. The resumed run reaches the **same final state as the uninterrupted reference** from section 2 (the dynamics are deterministic with `shots=0`).

In [ ]:
print(f"Reference final cut : {ref_final_cut:.6f}  (params {np.round(ref_params, 4)})")
print(f"Resumed   final cut : {resumed_final_cut:.6f}  (params {np.round(resumed_params, 4)})")
print(f"Final checkpoint iteration: {final_it}  (crash was at 12)")

# 1. It continued past the failure point rather than restarting from zero.
assert final_it == N_ITERS - 1, "resume did not reach the end"
assert final_it > 12, "resume did not get past the failure point"

# 2. The resumed run matches the uninterrupted reference exactly.
assert abs(resumed_final_cut - ref_final_cut) < 1e-6, "resumed state diverged from reference"
assert np.allclose(resumed_params, ref_params, atol=1e-6), "resumed params diverged"

print("\nPASS: the job resumed from its checkpoint and reached the same final")
print("state as an uninterrupted run -- the failure cost only the work since")
print("the last checkpoint, not the whole optimization.")

## 5. The same pattern as a real Hybrid Job

Locally we used a temp file and `json`. Inside a Braket Hybrid Job the mechanism is identical in spirit, but Braket manages the storage: `save_job_checkpoint()` persists a dict to the job's checkpoint location (S3-backed), and `load_job_checkpoint()` reads it back on a restart. Braket sets `AMZN_BRAKET_CHECKPOINT_DIR` and passes a `--checkpoint-config` so a relaunched job finds its prior state automatically.

Below is the entry-point *as a string* — it never runs in this notebook, so it never touches AWS. Notice the structure mirrors `run_recoverable` exactly: try to load a checkpoint, resume or start fresh, then checkpoint periodically and log a metric each step.

In [ ]:
QAOA_CHECKPOINT_ENTRY_POINT = r'''
import os
import numpy as np
from braket.circuits import Circuit, FreeParameter
from braket.devices import LocalSimulator
from braket.jobs import save_job_checkpoint, load_job_checkpoint, save_job_result
from braket.jobs.metrics import log_metric


def main():
    job_name = os.environ["AMZN_BRAKET_JOB_NAME"]
    n_iters = int(os.environ.get("AMZN_BRAKET_HP_iterations", "100"))
    checkpoint_every = int(os.environ.get("AMZN_BRAKET_HP_checkpoint_every", "10"))

    device = LocalSimulator()  # or AwsDevice(os.environ["AMZN_BRAKET_DEVICE_ARN"])
    gamma, beta = FreeParameter("gamma"), FreeParameter("beta")
    circuit = (Circuit().h(0).h(1).zz(0, 1, 2 * gamma)
               .rx(0, 2 * beta).rx(1, 2 * beta).probability())

    def cut_value(p):
        r = device.run(circuit, shots=0,
                       inputs={"gamma": float(p[0]), "beta": float(p[1])}).result()
        v = r.values[0]
        return v[1] + v[2]

    def grad(p, eps=1e-3):
        g = np.zeros(2)
        for i in range(2):
            s = np.zeros(2); s[i] = eps
            g[i] = (-cut_value(p + s) + cut_value(p - s)) / (2 * eps)
        return g

    # --- Resume from the last checkpoint if one exists ---
    try:
        state = load_job_checkpoint(job_name)
        start = state["iteration"] + 1
        params = np.array(state["params"])
        print(f"Resumed from checkpoint at iteration {state['iteration']}")
    except Exception:
        start, params = 0, np.array([0.1, 0.1])
        print("No checkpoint found; starting fresh")

    for it in range(start, n_iters):
        params = params - 0.3 * grad(params)
        log_metric(metric_name="cut_value", value=float(cut_value(params)),
                   iteration_number=it)
        if it % checkpoint_every == 0:
            save_job_checkpoint(
                checkpoint_data={"iteration": it, "params": params.tolist()},
                job_name=job_name,
            )

    save_job_result({"optimal_params": params.tolist(),
                     "final_cut": float(cut_value(params))})


if __name__ == "__main__":
    main()
'''

# Write the entry-point to a temp file (no AWS, no cost) so create() could point at it.
entry_dir = tempfile.mkdtemp(prefix="qaoa_job_")
entry_path = os.path.join(entry_dir, "qaoa_checkpoint_job.py")
with open(entry_path, "w") as f:
    f.write(QAOA_CHECKPOINT_ENTRY_POINT)

# Sanity-check that the entry-point at least parses (still no AWS).
compile(QAOA_CHECKPOINT_ENTRY_POINT, entry_path, "exec")
print(f"Entry-point written and compiles cleanly:\n  {entry_path}")

### Submitting the job (gated behind `RUN_ON_AWS`)

The call below would create a real, billable Hybrid Job. It stays gated behind `RUN_ON_AWS = False` so this notebook never touches AWS. To run it for real, deploy infra (`make deploy-infra`), set `RUN_ON_AWS = True`, and provide an S3 bucket.

**COST NOTE:** A Hybrid Job bills the classical instance per hour (`ml.m5.large` ~ $0.10-$0.30/hr) for as long as the container runs, plus the usual per-task / per-shot quantum charges if it targets a real QPU. Always prototype on `LocalSimulator` (as above) first, cap duration with a `stopping_condition` (`maxRuntimeInSeconds`), and let checkpointing make a relaunch cheap. The `checkpoint_config` is what lets a relaunched job pick up where a failed one left off.

In [ ]:
RUN_ON_AWS = False  # Leave False. Set True only with credentials + an S3 bucket (incurs cost).

if RUN_ON_AWS:
    from braket.jobs.config import CheckpointConfig

    job = AwsQuantumJob.create(
        device="local:braket/braket.local.qubit",  # or a QPU ARN
        source_module=entry_dir,
        entry_point="qaoa_checkpoint_job:main",
        job_name="qaoa-checkpoint-demo",
        hyperparameters={"iterations": "100", "checkpoint_every": "10"},
        stopping_condition={"maxRuntimeInSeconds": 3600},  # cap wall-clock -> cap cost
        # checkpoint_config points a relaunch at the prior job's saved state:
        checkpoint_config=CheckpointConfig(
            s3Uri="s3://YOUR-BUCKET/qaoa-checkpoints",
            localPath="/opt/jobs/checkpoints",
        ),
    )
    print("Submitted:", job.arn)
    print("Final result:", job.result())
else:
    print("RUN_ON_AWS is False -- no job submitted, no cost.")
    print("The local demo above already proved the resume logic end-to-end.")

## 6. The checkpoint-interval vs. overhead trade-off

Checkpointing is not free: each save is an I/O round-trip (to S3 in a real job). The interval is a dial between two costs:

- **Checkpoint rarely** (large interval): cheap I/O, but a failure throws away all the work done *since* the last checkpoint. Worst case lost work approaches one full interval.
- **Checkpoint every step** (interval = 1): nothing is ever lost on failure, but you pay an I/O write on every single iteration.

The right interval depends on how expensive an iteration is, how expensive a checkpoint write is, and how likely a failure is. We estimate both costs across a range of intervals for our 100-iteration job, assuming a failure occurs at a random iteration.

In [ ]:
n = 100                 # iterations in the (hypothetical) full job
iter_cost = 1.0         # arbitrary compute units per iteration
ckpt_io_cost = 2.0      # units per checkpoint write (S3 round-trip is not free)

intervals = np.arange(1, 26)
# Expected wasted iterations on a uniformly-random failure ~ (interval - 1) / 2.
expected_wasted = (intervals - 1) / 2.0 * iter_cost
# Total checkpoint I/O paid over a full successful run.
total_io = np.ceil(n / intervals) * ckpt_io_cost
total_overhead = expected_wasted + total_io
best = intervals[int(np.argmin(total_overhead))]

plt.figure(figsize=(7, 4))
plt.plot(intervals, expected_wasted, "o-", label="expected work lost on failure")
plt.plot(intervals, total_io, "s-", label="total checkpoint I/O")
plt.plot(intervals, total_overhead, "^-", color="#111827", label="combined overhead")
plt.axvline(best, ls="--", color="#9ca3af", label=f"sweet spot ~ every {best}")
plt.xlabel("checkpoint interval (iterations)")
plt.ylabel("cost (arbitrary units)")
plt.title("Checkpoint too often: I/O dominates. Too rarely: lost work dominates.")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Lowest combined overhead near interval = {best} for these cost assumptions.")

## Exercises

Work through these locally on the `LocalSimulator` — none require AWS.

In [ ]:
# Exercise 1: Checkpoint the FULL optimizer state.
# Real optimizers (Adam, momentum) carry running moments. Extend the checkpoint
# dict to persist a velocity/moment vector alongside params, and prove a resumed
# momentum-based run still matches an uninterrupted one.
# TODO: add a 'velocity' field to save_checkpoint / load_checkpoint and to the loop.


# Exercise 2: Two failures.
# Make run_recoverable fail at iteration 8 AND again at iteration 15, then resume
# a third time. Assert the final state still matches the reference.
# TODO: call run_recoverable with different fail_at values across attempts.


# Exercise 3: Corrupted checkpoint.
# Truncate the checkpoint file mid-write, then attempt to load it. Add a guard
# that falls back to the previous-good checkpoint (keep two: current + previous).
# TODO: write save_checkpoint_atomic() that writes to a .tmp then os.replace().


# Exercise 4: Empirical sweet spot.
# Instead of the analytic curve in section 6, Monte-Carlo it: for each interval,
# sample 1000 random failure iterations and measure mean wasted + I/O. Compare
# the empirical minimum to the analytic one.
# TODO: use rng.integers(0, n) to sample failure points.
print("Exercises ready -- edit and run.")

## Summary

- A long-running optimization should treat its **optimizer state** (parameters + iteration, and any optimizer moments) as the asset to protect against mid-run failure.
- Checkpointing is a **save-then-resume** loop: on start, try to load the last checkpoint and continue from `last_iteration + 1`; otherwise start fresh. We proved locally that a run which crashed at iteration 12 resumed and reached the *same final state* as an uninterrupted run.
- Locally we used a `tempfile` + `json`; a real Hybrid Job uses `save_job_checkpoint()` / `load_job_checkpoint()`, with Braket persisting state to S3 and a `CheckpointConfig` letting a relaunch find it. Those AWS calls live only inside the gated entry-point — nothing here touched AWS.
- The **checkpoint interval** trades recovery granularity against I/O overhead: too rare loses work on failure, too frequent pays I/O every step. The sweet spot depends on iteration cost, checkpoint cost, and failure rate.

**Next:** [`05-custom-containers.ipynb`](05-custom-containers.ipynb) — package heavier dependencies (the chemistry stack from module 05) into a custom container and run a job on your own image.